Pre process

In [1]:
import pandas as pd
import numpy as np
import math
import torch

In [2]:
df=pd.read_csv("./datasets/cleaned_data.csv")
df.drop(columns=["sn"], inplace=True)
df["month"]=df["month"].map({
    1:"january",
    2:"february",
    3:"march",
    4:"april",
    5:"may",
    6:"june",
    7:"july",
    8:"august",
    9:"september",
    10:"october",
    11:"november",
    12:"december"
})

In [3]:
month_order="january,february,march,april,may,june,july,august,september,october,november,december".split(",")
df["month"]=pd.Categorical(df["month"], categories=month_order, ordered=True)

item_order=[i+1 for i in range(100)]
df["item_id"]=pd.Categorical(df["item_id"], categories=item_order, ordered=True)

month=pd.get_dummies(df["month"], prefix="month", dtype=int)
item_id=pd.get_dummies(df["item_id"], prefix="item_id", dtype=int)

df.drop(columns=["month", "item_id", "date", "name", "year","kcal", "price"], inplace=True)
df=pd.concat([df,item_id, month], axis=1)

df["item_count"] = df["item_count"].apply(np.log1p)

df.drop(columns=["day"], inplace=True)

df["item_count"]=(df["item_count"]-df["item_count"].min())/(df["item_count"].max()-df["item_count"].min())


df=df.sample(frac=1, random_state=42)

df.head()

,item_count,item_id_1,item_id_2,item_id_3,item_id_4,item_id_5,item_id_6,item_id_7,item_id_8,item_id_9,...,month_march,month_april,month_may,month_june,month_july,month_august,month_september,month_october,month_november,month_december
44479,0.000000,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
70455,0.000000,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
56897,0.000000,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
72117,0.560128,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
54651,0.000000,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0


split train and test

In [4]:
def split_train_test(x,y,size):
    train_len=math.floor(len(y)*size)
    
    x_train=torch.tensor(x[0:train_len], dtype=torch.float32, device="cuda", requires_grad=True)
    y_train=torch.tensor(y[0:train_len], dtype=torch.float32, device="cuda")
    
    x_test=torch.tensor(x[train_len:], dtype=torch.float32, device="cuda", requires_grad=True)
    y_test=torch.tensor(y[train_len:], dtype=torch.float32, device="cuda")
    
    return x_train,y_train,x_test,y_test

In [5]:
x_train,y_train,x_test,y_test = split_train_test(x=df.drop(columns=["item_count"]).values, y=(df["item_count"].values), size=0.8)

In [9]:
class Model :
    
    def __init__(self, x, y, neurons=[8,8], device="cuda"):
        self.x = x.to(device)
        self.y = y.view(-1,1).to(device)
        self.device=device
        #shape of x
        N,d = x.shape
        #weight and bias of first
        self.w1 = (torch.randn(d, neurons[0], dtype=torch.float32, device=device)*(2/d)**0.5).requires_grad_()
        self.b1 = torch.randn(neurons[0], dtype=torch.float32, device=device, requires_grad=True)
        #weight and bias of second neurns
        self.w2 = (torch.randn(neurons[0], neurons[1], dtype=torch.float32, device=device)*(2/neurons[0])**0.5).requires_grad_()
        self.b2 = torch.randn(neurons[1], dtype=torch.float32, device=device, requires_grad=True)
        #weight and bias of third neurons
        self.w3 = (torch.randn(neurons[1], 1, dtype=torch.float32, device=device)*(2/neurons[1])**0.5).requires_grad_()
        self.b3 = torch.randn(1, dtype=torch.float32, device=device, requires_grad=True)
    def train(self, epochs=1000, batch_size=1, lr=0.01):
        
        for epoch in range(epochs):
            curr_batch=0
            row_len = self.x.shape[0]
            
            total_loss = 0
            
            while(curr_batch < row_len):
                
                curr_batch_len=batch_size
                if row_len-curr_batch >= batch_size : 
                    curr_batch_len=batch_size
                else:
                    curr_batch_len = row_len-curr_batch
                
                x_batch = self.x[curr_batch:curr_batch+curr_batch_len]
                y_batch = self.y[curr_batch:curr_batch+curr_batch_len]
                
                n1 = torch.relu( x_batch @ self.w1 + self.b1)
                n2 = torch.relu(n1 @ self.w2 + self.b2)
                n3 = n2 @ self.w3 + self.b3
                
                error = n3 - y_batch
                
                mse = (error**2).mean()
                
                total_loss += mse.item() * curr_batch_len
                
                mse.backward()
                
                with torch.no_grad():
                    self.w1 -= lr*self.w1.grad
                    self.b1 -= lr*self.b1.grad
                    self.w2 -= lr*self.w2.grad
                    self.b2 -= lr*self.b2.grad
                    self.w3 -= lr*self.w3.grad
                    self.b3 -= lr*self.b3.grad
                
                self.w1.grad.zero_()
                self.b1.grad.zero_()
                self.w2.grad.zero_()
                self.b2.grad.zero_()
                self.w3.grad.zero_()
                self.b3.grad.zero_()
                
                curr_batch+=curr_batch_len
            
            if (epoch+1)%(1000)==0:
                print(f"Epoch====({epoch+1}): Loss={(total_loss/row_len)}")
    
    def predict(self, x_new):
        x_new = x_new.to(self.device)
        n1 = torch.relu(x_new @ self.w1 + self.b1)
        n2 = torch.relu(n1 @ self.w2 + self.b2 )
        n3 = n2 @ self.w3 + self.b3 
        return n3

In [10]:
model = Model(x_train, y_train, neurons=[256, 256], device="cuda")

model.train(epochs=50000, batch_size=4096, lr=0.0003)

Epoch====(1000): Loss=0.020845064591534818
Epoch====(2000): Loss=0.0150000799460894
Epoch====(3000): Loss=0.01180490155289643
Epoch====(4000): Loss=0.009923929999833995
Epoch====(5000): Loss=0.008801225715581952
Epoch====(6000): Loss=0.008137081522684898
Epoch====(7000): Loss=0.007733169357788607
Epoch====(8000): Loss=0.0074757490375072415
Epoch====(9000): Loss=0.007305727226754827
Epoch====(10000): Loss=0.0071888493183646755
Epoch====(11000): Loss=0.00710160456339482
Epoch====(12000): Loss=0.0070344636563456405
Epoch====(13000): Loss=0.006980426011974142
Epoch====(14000): Loss=0.006933717666887236
Epoch====(15000): Loss=0.006895101771275275
Epoch====(16000): Loss=0.006861062100442657
Epoch====(17000): Loss=0.006830735850644155
Epoch====(18000): Loss=0.0068029208210752395
Epoch====(19000): Loss=0.006777655212276608
Epoch====(20000): Loss=0.006754211528756975
Epoch====(21000): Loss=0.006732418093096166
Epoch====(22000): Loss=0.006712053756970559
Epoch====(23000): Loss=0.0066931113655114

In [11]:
y_pred = model.predict(x_test)

y_pred_real = torch.expm1(y_pred)

y_pred_real[0:10]

tensor([[ 0.0065],
        [ 0.1399],
        [ 0.0080],
        [ 0.0612],
        [-0.0153],
        [-0.0029],
        [ 0.0011],
        [ 0.0083],
        [-0.0004],
        [ 0.2125]], device='cuda:0', grad_fn=<SliceBackward0>)

In [12]:
def r2_score(y_true, y_pred):
    # make sure y_true and y_pred are the same shape
    y_true = y_true.view(-1, 1)
    y_pred = y_pred.view(-1, 1)
    
    # total sum of squares
    ss_tot = ((y_true - y_true.mean())**2).sum()
    
    # residual sum of squares
    ss_res = ((y_true - y_pred)**2).sum()
    
    r2 = 1 - ss_res / ss_tot
    return r2.item()

In [13]:
accuracy = r2_score(y_test, y_pred)
print(f"R² = {accuracy:.4f}")  

R² = 0.7968


Export data

In [ ]:
import torch.nn as nn

class ONNXWrapper(nn.Module):
    def __init__(self, manual_model):
        super().__init__()
        self.model = manual_model

    def forward(self, x):
        # just call your manual predict function
        return self.model.predict(x)

# --- 2. Create wrapper instance ---
onnx_model = ONNXWrapper(model)  # 'model' is your trained Model instance
onnx_model.eval()  # set to eval mode

# --- 3. Create a dummy input with same shape as your x_train ---
dummy_input = torch.randn(1, x_train.shape[1], device="cuda")  # 1 row, all features

# --- 4. Export to ONNX ---
torch.onnx.export(
    onnx_model, 
    dummy_input, 
    "sales_prediction.onnx",
    input_names=["input"], 
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=17
)

print("ONNX model exported successfully!")

C:\Users\tikad\AppData\Local\Temp\ipykernel_20040\625406965.py:20: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0302 19:50:52.194000 20040 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ONNXWrapper()` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ONNXWrapper()` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


c:\Projects\machine_learning\demand_forecasting\model_training\env\Lib\site-packages\torch\export\_unlift.py:825: UserWarning: A model attribute `lifted_tensor_0` requires gradient. but it's not properly registered as a parameter. torch.export will detach it and treat it as a constant tensor but please register it as parameter instead.
  unlift_gm = _create_stateful_graph_module(new_gm, ep.range_constraints, ep)
c:\Projects\machine_learning\demand_forecasting\model_training\env\Lib\site-packages\torch\export\_unlift.py:825: UserWarning: A model attribute `lifted_tensor_1` requires gradient. but it's not properly registered as a parameter. torch.export will detach it and treat it as a constant tensor but please register it as parameter instead.
  unlift_gm = _create_stateful_graph_module(new_gm, ep.range_constraints, ep)
c:\Projects\machine_learning\demand_forecasting\model_training\env\Lib\site-packages\torch\export\_unlift.py:825: UserWarning: A model attribute `lifted_tensor_2` requi

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
ONNX model exported successfully!
